# 🎓 DocuAgent de arriba abajo — curso completo de LangChain/LangGraph sobre tu propio agente

Nada de ejemplos de juguete: cada concepto se aprende **construyendo y desmontando DocuAgent**, tu analista de configuraciones RAG sobre el observatorio. Al terminar sabrás exactamente qué hace el framework por ti y qué harías tú a mano.

| Módulo | Concepto | Dónde vive en TU código |
|---|---|---|
| 1 | Mensajes y chat models en crudo | `fabricas.py` |
| 2 | LCEL y salida estructurada | `corpus.py` (preguntas de test) |
| 3 | Tools + el bucle ReAct **desmontado a mano** | — (la lección clave) |
| 4 | El mismo bucle como `StateGraph` construido por ti | `bancos/base.py`, `meta.py` |
| 5 | Los tres modos de streaming | `meta.iniciar/reanudar` |
| 6 | Memoria conversacional y gestión del contexto | checkpointer del `meta` |
| 7 | Checkpoints: historia, **edición de estado**, forks | `MemorySaver` del `meta` |
| 8 | HITL con `interrupt()` sobre DocuAgent | las 4 pausas de `meta.py` |
| 9 | Evaluación sistemática del agente | tu propia filosofía de lab |

⚠️ **Coste**: usa el modelo barato de la celda 0 para estudiar; el bucle se comporta igual.

## 0 · Setup

In [ ]:
from rag_lab.memory import Memoria
from rag_lab.analisis import observatorio as obs

mem = Memoria()
print("Evidencia disponible:", obs.panorama(mem))

# Modelo del agente. Para ESTUDIAR el bucle, el barato; para análisis serio, el juez.
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)   # barato
# from rag_lab.fabricas import construir_llm_juez; llm = construir_llm_juez()  # serio

---
## 1 · Primitivas: mensajes y chat models en crudo

Debajo de todo agente hay una sola operación: **una lista de mensajes entra, un `AIMessage` sale**. Sin tools, el modelo solo sabe lo que le pegues en el prompt — compruébalo dándole tú mismo un dato del observatorio.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

panorama = obs.panorama(mem)
mensajes = [
    SystemMessage(content="Eres un analista escueto de experimentos RAG."),
    HumanMessage(content=f"Con estos datos: {panorama}\n¿Cuánta evidencia hay y de qué bancos?"),
]
respuesta = llm.invoke(mensajes)
print(type(respuesta).__name__, "->", respuesta.content)
print("\nuso de tokens:", respuesta.usage_metadata)

Fíjate: `respuesta` es un `AIMessage` con `content` y `usage_metadata`. Todo lo demás del curso (tools, grafos, streaming) son formas de **fabricar y encadenar estas listas de mensajes** automáticamente.

---
## 2 · LCEL y salida estructurada

El operador `|` compone runnables: `prompt | llm` es un pipeline. Y `with_structured_output` obliga al modelo a devolver un Pydantic — es EXACTAMENTE como `corpus.py` genera tus preguntas de test (`ListaPreguntasTest`). Aquí lo usamos para algo muy DocuAgent: clasificar si una afirmación analítica está soportada por datos.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class AuditoriaAfirmacion(BaseModel):
    soportada: bool = Field(description="¿La afirmación se sigue de los datos?")
    problema: str = Field(description="Si no, qué falla (causalidad inventada, "
                                      "n insuficiente, mezcla de escalas...)")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Auditas afirmaciones sobre experimentos RAG. Sé estricto: "
               "correlación no es causalidad y n<5 es anécdota."),
    ("human", "DATOS: {datos}\nAFIRMACIÓN: {afirmacion}"),
])
auditor = prompt | llm.with_structured_output(AuditoriaAfirmacion)

v = auditor.invoke({
    "datos": "bge-m3 media=0.62 (n=3 documentos); granite media=0.60 (n=3)",
    "afirmacion": "bge-m3 es superior porque su arquitectura entiende mejor el español",
})
print(v)

Dos lecciones: el pipeline es un runnable más (se puede `invoke`, `stream`, `batch`), y la salida estructurada convierte al LLM en una **función tipada** — la base de todo lo verificable en tu lab (preguntas, veredictos, la capa 2 de `verificar_preguntas`).

---
## 3 · Tools — y el bucle ReAct desmontado a mano

**La celda más importante del curso.** Antes de que LangGraph lo automatice, ejecuta TÚ el ciclo completo: el modelo pide una tool → tú la ejecutas → le devuelves la observación → responde. Cuando veas al framework hacerlo, sabrás exactamente qué está pasando.

In [ ]:
from langchain_core.tools import tool


@tool
def rendimiento_de_etiqueta(banco: str, etiqueta: str) -> str:
    """Rendimiento agregado de UNA config: media + IC bootstrap sobre todas sus
    mediciones históricas del observatorio."""
    r = obs.rendimiento(mem, banco, etiqueta)
    if not r.get("disponible"):
        return f"Sin mediciones de '{etiqueta}' en {banco}."
    return f"{etiqueta}: n={r['n']} media={r['media']} IC=[{r['ic_bajo']}, {r['ic_alto']}]"


@tool
def duelo(banco: str, a: str, b: str) -> str:
    """Comparación PAREADA entre dos configs, con desglose por documento. Única
    fuente válida para afirmar que una config gana a otra."""
    d = obs.duelo(mem, banco, a, b)
    if not d.get("disponible"):
        return f"Evidencia insuficiente: {d.get('motivo')}"
    g = d["global"]
    return (f"{a} vs {b}: dif={g['dif_media']:+} IC=[{g['ic_bajo']}, {g['ic_alto']}] "
            f"CONCLUYENTE={g['concluyente']} | por doc: {d['por_documento']}")


TOOLS = [rendimiento_de_etiqueta, duelo]

# 1) Las tools son funciones normales: pruébalas SIN modelo
print("directo:", rendimiento_de_etiqueta.invoke({"banco": "embedding", "etiqueta": "bge-m3"}))
# 2) y tienen un esquema que es lo ÚNICO que el modelo ve de ellas:
print("\nesquema que ve el modelo:", duelo.args)

In [ ]:
from langchain_core.messages import ToolMessage

llm_con_tools = llm.bind_tools(TOOLS)          # el modelo ahora PUEDE pedir tools
historial = [
    SystemMessage(content="Analista RAG. Todo número debe salir de una herramienta."),
    HumanMessage(content="¿Cómo rinde bge-m3 en el banco de embeddings?"),
]

# --- vuelta 1: el modelo NO responde; PIDE una tool ---
paso1 = llm_con_tools.invoke(historial)
print("tool_calls:", paso1.tool_calls)
historial.append(paso1)

# --- TÚ ejecutas la tool y devuelves la observación como ToolMessage ---
por_nombre = {t.name: t for t in TOOLS}
for tc in paso1.tool_calls:
    salida = por_nombre[tc["name"]].invoke(tc["args"])
    print("observación:", salida)
    historial.append(ToolMessage(content=str(salida), tool_call_id=tc["id"]))

# --- vuelta 2: con la observación en el historial, ahora sí responde ---
paso2 = llm_con_tools.invoke(historial)
print("\nRESPUESTA FINAL:", paso2.content)

Eso es TODO el secreto de un agente: `bind_tools` + un bucle `mientras haya tool_calls: ejecutar y devolver ToolMessage`. El `tool_call_id` empareja petición y observación. Todo lo que viene ahora es este bucle con mejores ropajes.

---
## 4 · El mismo bucle como `StateGraph` construido POR TI

Nada de `create_react_agent`: montas el grafo pieza a pieza, como hace tu `meta.py`. Ingredientes: un **estado** con reducer (`MessagesState` acumula mensajes en vez de sobrescribirlos), un nodo agente, un `ToolNode` que ejecuta el bucle de la celda anterior por ti, y una **arista condicional** que decide si seguir o terminar.

In [ ]:
def dibujar(grafo) -> None:
    """ASCII si grandalf está instalado; si no, Mermaid (sin dependencias).
    Pega el Mermaid en https://mermaid.live para verlo dibujado."""
    try:
        print(grafo.get_graph().draw_ascii())
    except ImportError:
        print("(grandalf no instalado: `pip install grandalf` para ASCII; "
              "mientras, Mermaid:)\n")
        print(grafo.get_graph().draw_mermaid())


from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

SISTEMA = ("Eres DocuAgent, analista de experimentos RAG. Todo número sale de una "
           "herramienta; 'gana' solo si el duelo dice CONCLUYENTE=True; n siempre visible.")


def nodo_agente(estado: MessagesState) -> dict:
    """Un nodo = una función estado -> cambios. Igual que en bancos/base.py."""
    mensajes = [SystemMessage(content=SISTEMA)] + estado["messages"]
    return {"messages": [llm_con_tools.invoke(mensajes)]}   # el reducer ACUMULA


constructor = StateGraph(MessagesState)
constructor.add_node("agente", nodo_agente)
constructor.add_node("tools", ToolNode(TOOLS))
constructor.add_edge(START, "agente")
constructor.add_conditional_edges("agente", tools_condition)  # ¿tool_calls? tools : END
constructor.add_edge("tools", "agente")                       # y vuelta al agente

checkpointer = MemorySaver()
docuagent = constructor.compile(checkpointer=checkpointer)
dibujar(docuagent)

In [ ]:
cfg = {"configurable": {"thread_id": "curso-1"}}
r = docuagent.invoke({"messages": [HumanMessage(content=
    "¿Hay diferencia real entre bge-m3 y granite-311m-r2 en embeddings?")]}, cfg)
print(r["messages"][-1].content)

Lo que acabas de construir es funcionalmente `create_react_agent(llm, TOOLS, prompt=SISTEMA)` — pero ahora sabes qué hay dentro: `tools_condition` mira si el último mensaje trae `tool_calls`, y `ToolNode` es la celda del módulo 3 empaquetada. Tu `meta.py` es esto mismo con más nodos, rutas propias y pausas.

---
## 5 · Los tres modos de streaming

La misma pregunta, tres lupas: `values` (el estado completo tras cada paso), `updates` (solo lo que cambió — la de las trazas), y `messages` (**token a token**, para UIs).

In [ ]:
pregunta = {"messages": [HumanMessage(content="¿Cómo rinde granite-97m-r2 en embeddings?")]}

print("=== updates: qué añadió cada nodo ===")
for ev in docuagent.stream(pregunta, {"configurable": {"thread_id": "s-upd"}},
                           stream_mode="updates"):
    for nodo, cambio in ev.items():
        tipos = [type(m).__name__ for m in cambio.get("messages", [])]
        print(f"  [{nodo}] -> {tipos}")

print("\n=== values: tamaño del estado tras cada paso ===")
for ev in docuagent.stream(pregunta, {"configurable": {"thread_id": "s-val"}},
                           stream_mode="values"):
    print(f"  estado con {len(ev['messages'])} mensajes")

print("\n=== messages: token a token (los primeros 40 trozos) ===")
n = 0
for trozo, meta_info in docuagent.stream(pregunta, {"configurable": {"thread_id": "s-tok"}},
                                         stream_mode="messages"):
    if trozo.content and n < 40:
        print(trozo.content, end="", flush=True)
        n += 1
print(" ...")

---
## 6 · Memoria conversacional y gestión del contexto

El checkpointer + `thread_id` dan memoria multi-turno gratis: el segundo turno **recuerda** el primero. El precio: el historial crece y con él los tokens de entrada — míralo crecer, y aprende a podarlo.

In [ ]:
cfg = {"configurable": {"thread_id": "conversacion-1"}}

r1 = docuagent.invoke({"messages": [HumanMessage(content=
    "¿Cómo rinde bge-m3 en embeddings?")]}, cfg)
print("T1:", r1["messages"][-1].content[:150], "...")
print("    tokens entrada:", r1["messages"][-1].usage_metadata["input_tokens"])

# El pronombre 'lo' solo tiene sentido si RECUERDA el turno anterior:
r2 = docuagent.invoke({"messages": [HumanMessage(content=
    "¿Y cómo se compara con granite-311m-r2? Enfréntalos.")]}, cfg)
print("\nT2:", r2["messages"][-1].content[:150], "...")
print("    tokens entrada:", r2["messages"][-1].usage_metadata["input_tokens"])
print("    mensajes acumulados en el hilo:", len(r2["messages"]))

In [ ]:
# Poda de contexto: variante del nodo agente que solo mira los últimos N mensajes.
# (En producción usarías trim_messages de langchain_core; la idea es la misma.)
VENTANA = 6

def nodo_agente_podado(estado: MessagesState) -> dict:
    recientes = estado["messages"][-VENTANA:]
    # nunca cortar por en medio de un ciclo tool_call/ToolMessage:
    while recientes and type(recientes[0]).__name__ == "ToolMessage":
        recientes = recientes[1:]
    return {"messages": [llm_con_tools.invoke(
        [SystemMessage(content=SISTEMA)] + recientes)]}

c2 = StateGraph(MessagesState)
c2.add_node("agente", nodo_agente_podado)
c2.add_node("tools", ToolNode(TOOLS))
c2.add_edge(START, "agente")
c2.add_conditional_edges("agente", tools_condition)
c2.add_edge("tools", "agente")
docuagent_podado = c2.compile(checkpointer=MemorySaver())
print("Variante con ventana de contexto lista: mismo grafo, nodo agente distinto.")

---
## 7 · Checkpoints a fondo: historia, **edición de estado** y forks

Ya viste `get_state_history` en el otro notebook. La pieza nueva y potente es **`update_state`**: editar la memoria del agente a mano. Le inyectamos una observación falsa y vemos cómo su respuesta cambia — la demostración definitiva de que el agente ES su historial de mensajes.

In [ ]:
cfg = {"configurable": {"thread_id": "edicion-1"}}
r = docuagent.invoke({"messages": [HumanMessage(content=
    "¿Cómo rinde bge-m3 en embeddings?")]}, cfg)
print("ANTES:", r["messages"][-1].content[:200])

# Inyectamos en su memoria un mensaje humano que corrige el rumbo:
docuagent.update_state(cfg, {"messages": [HumanMessage(content=
    "Ojo: los datos de bge-m3 anteriores a julio son de otra era de métricas; "
    "acláralo en tu siguiente respuesta.")]})

r2 = docuagent.invoke({"messages": [HumanMessage(content=
    "Resume tu análisis de bge-m3 en una frase.")]}, cfg)
print("\nDESPUÉS (con el estado editado):", r2["messages"][-1].content)

`update_state` es también la mecánica interna de las decisiones HITL de tu `meta`: reanudar con `Command(resume=...)` no es más que escribir la decisión en el estado y continuar. Y cada `update_state` crea un checkpoint nuevo — la historia se puede recorrer y bifurcar (`get_state_history` + invocar desde un checkpoint viejo, como en `docuagent_estudio.ipynb` §6).

---
## 8 · HITL: DocuAgent pide permiso antes de gastar

La joya de tu `meta.py`, replicada aquí en pequeño: un nodo-peaje con `interrupt()` **antes** de ejecutar tools caras. El grafo se congela, tú decides, `Command(resume=...)` continúa. Compara con `_pausa` de tu meta: es el mismo patrón.

In [ ]:
from langgraph.types import Command, interrupt

TOOLS_CARAS = {"duelo"}    # imagina que el duelo re-evalúa pipelines y cuesta dinero


def nodo_peaje(estado: MessagesState) -> dict:
    """Se interpone entre agente y tools: si la tool pedida es cara, pausa."""
    ultimo = estado["messages"][-1]
    caras = [tc for tc in (getattr(ultimo, "tool_calls", None) or [])
             if tc["name"] in TOOLS_CARAS]
    if caras:
        decision = interrupt({
            "mensaje": "DocuAgent quiere ejecutar tools caras. ¿Autorizas?",
            "tools": [f"{tc['name']}({tc['args']})" for tc in caras],
            "como_responder": {"autorizar": {}, "denegar": {"denegar": True}},
        })
        if decision.get("denegar"):
            return {"messages": [ToolMessage(
                content="DENEGADO por el humano: no ejecutes esta herramienta; "
                        "responde con lo que ya tengas y dilo explícitamente.",
                tool_call_id=caras[0]["id"])]}
    return {}


c3 = StateGraph(MessagesState)
c3.add_node("agente", nodo_agente)
c3.add_node("peaje", nodo_peaje)
c3.add_node("tools", ToolNode(TOOLS))
c3.add_edge(START, "agente")
c3.add_conditional_edges("agente", tools_condition, {"tools": "peaje", END: END})
c3.add_conditional_edges("peaje",
    lambda e: "tools" if getattr(e["messages"][-1], "tool_calls", None) else "agente")
c3.add_edge("tools", "agente")
docuagent_hitl = c3.compile(checkpointer=MemorySaver())
dibujar(docuagent_hitl)

In [ ]:
cfg = {"configurable": {"thread_id": "hitl-1"}}
estado = docuagent_hitl.invoke({"messages": [HumanMessage(content=
    "Enfrenta bge-m3 contra granite-311m-r2 en embeddings.")]}, cfg)

if estado.get("__interrupt__"):
    print("PAUSA:", estado["__interrupt__"][0].value, "\n")
    # decide: {"seguir": True} autoriza; {"denegar": True} deniega.
    # (nunca un resume FALSY como {}: puede tratarse como "sin resume" y re-pausar)
    estado = docuagent_hitl.invoke(Command(resume={"seguir": True}), cfg)
print("FINAL:", estado["messages"][-1].content[:400])

---
## 9 · Evaluar al agente como evalúas los RAG

Cierre con tu propia filosofía: la verdad la da el **observatorio** (determinista), y el agente aprueba si su respuesta la respeta. Una mini-batería auto-corregida: para cada duelo, calculamos la verdad con `obs.duelo` y comprobamos que la prosa del agente no la contradice.

In [ ]:
def evaluar_agente(banco: str, a: str, b: str) -> dict:
    verdad = obs.duelo(mem, banco, a, b)
    cfg = {"configurable": {"thread_id": f"eval-{a}-{b}"}}
    r = docuagent.invoke({"messages": [HumanMessage(content=
        f"¿Hay diferencia concluyente entre {a} y {b} en el banco {banco}? "
        f"Responde citando el veredicto de la comparación.")]}, cfg)
    texto = r["messages"][-1].content.lower()
    if not verdad.get("disponible"):
        correcto = "insuficiente" in texto or "sin" in texto
        esperado = "declarar evidencia insuficiente"
    elif verdad["global"]["concluyente"]:
        correcto = "concluyente" in texto and " no concluyente" not in texto
        esperado = "afirmar diferencia concluyente"
    else:
        correcto = "empate" in texto or "no concluyente" in texto or "no es concluyente" in texto
        esperado = "declarar empate/no concluyente"
    return {"duelo": f"{a} vs {b}", "esperado": esperado,
            "correcto": correcto, "respuesta": r["messages"][-1].content[:160]}


baterias = [("embedding", "bge-m3", "granite-311m-r2"),
            ("embedding", "bge-m3", "voyage-3-lite"),
            ("embedding", "qwen3-0.6b", "granite-97m-r2")]
aciertos = 0
for banco, a, b in baterias:
    res = evaluar_agente(banco, a, b)
    aciertos += res["correcto"]
    print(f"{'✓' if res['correcto'] else '✗'} {res['duelo']} | esperado: {res['esperado']}")
    print(f"    {res['respuesta']}...\n")
print(f"Batería: {aciertos}/{len(baterias)}")

---
## 10 · Mapa final: concepto → este notebook → tu código → doc oficial

| Concepto | Aquí | En rag_lab | Doc |
|---|---|---|---|
| Mensajes / chat models | §1 | `fabricas.py` | docs.langchain.com → Messages |
| LCEL (`prompt \| llm`) | §2 | `corpus.py` (generación de preguntas) | → LCEL |
| `with_structured_output` | §2 | `corpus.py`, `_PlanLLM` en `plan.py` | → Structured output |
| `@tool` / `bind_tools` / `ToolMessage` | §3 | tools de DocuAgent | → Tools |
| `StateGraph` + reducer + arista condicional | §4 | `bancos/base.py`, `meta.py` | → LangGraph Graph API |
| `ToolNode` / `tools_condition` | §4 | — | → LangGraph prebuilt |
| Streaming (3 modos) | §5 | `meta.iniciar` | → Streaming |
| Checkpointer / `thread_id` | §6 | `MemorySaver` del meta | → Persistence |
| `update_state` / forks | §7 | mecánica de `reanudar` | → Time travel |
| `interrupt()` / `Command(resume)` | §8 | `_pausa` y las 4 pausas del meta | → Human-in-the-loop |
| Evaluación con verdad determinista | §9 | toda la filosofía del lab | — |

**Itinerario sugerido**: corre §3 tres veces hasta que el bucle a mano te aburra (señal de que lo entiendes), reconstruye §4 sin mirar, y luego abre `meta.py` — lo leerás como quien lee su propio idioma. Los módulos de instrumentación fina (contador de tokens, time-travel con forks, inyección de fallos, grounding) siguen en `docuagent_estudio.ipynb`.